# Coincident Events

Two events flagged as coincident are stored in:
```
/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/i3files/NoDeepCore/HESE12/Bfr/coinc/
```

This notebook checks whether they appear in:
1. `HESE12.hdf5` — the reference HESE12 event selection
2. `EvtGen_merged.h5` — our v3 processing output

and prints a table of Run, Event, RecoETot, and HESE_CausalQTot for each.

In [2]:
import tables
import pandas as pd
import numpy as np

## Coincident event IDs

Extracted from the i3 files in the `coinc/` folder.

In [3]:
# (run, event) pairs from the two coincident i3 files
# taken from /data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/i3files/NoDeepCore/HESE12/Bfr/coinc/
COINC_EVENTS = [
    (131680, 66412090),  # Run00131680_MJD58422
    (132143, 36142391),  # Run00132143_MJD58519
]

HESE12_PATH   = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/hdf_files/NoDeepCore/HESE12/Bfr/HESE12.hdf5"
EVTGEN_PATH   = "/data/user/tvaneede/GlobalFit/reco_processing/data/hese/output/v3/merged/EvtGen_merged.h5"

## 1. HESE12.hdf5

In [5]:
with tables.open_file(HESE12_PATH, "r") as f:
    df_hese12 = pd.DataFrame({
        "Run":      f.root.I3EventHeader.col("Run"),
        "Event":    f.root.I3EventHeader.col("Event"),
        "RecoETot": f.root.RecoETot.col("value"),
        "QTot":     f.root.QTot.col("value"),
    })

print(f"HESE12.hdf5: {len(df_hese12)} events total")

HESE12.hdf5: 97 events total


In [6]:
coinc_set = set(COINC_EVENTS)
mask_hese12 = [(r, e) in coinc_set for r, e in zip(df_hese12["Run"], df_hese12["Event"])]
found_hese12 = df_hese12[mask_hese12].reset_index(drop=True)

print(f"Coincident events found in HESE12.hdf5: {len(found_hese12)} / {len(COINC_EVENTS)}")
for run, event in COINC_EVENTS:
    present = ((df_hese12["Run"] == run) & (df_hese12["Event"] == event)).any()
    print(f"  Run={run}, Event={event}: {'FOUND' if present else 'NOT FOUND'}")

Coincident events found in HESE12.hdf5: 2 / 2
  Run=131680, Event=66412090: FOUND
  Run=132143, Event=36142391: FOUND


## 2. EvtGen_merged.h5 (v3)

In [9]:
with tables.open_file(EVTGEN_PATH, "r") as f:
    df_evtgen = pd.DataFrame({
        "Run":             f.root.I3EventHeader.col("Run"),
        "Event":           f.root.I3EventHeader.col("Event"),
        "RecoETot":        f.root.RecoETot.col("value"),
        "HESE_CausalQTot": f.root.HESE_CausalQTot.col("value"),
    })

print(f"EvtGen_merged.h5: {len(df_evtgen)} events total")

EvtGen_merged.h5: 188 events total


In [10]:
mask_evtgen = [(r, e) in coinc_set for r, e in zip(df_evtgen["Run"], df_evtgen["Event"])]
found_evtgen = df_evtgen[mask_evtgen].reset_index(drop=True)

print(f"Coincident events found in EvtGen_merged.h5: {len(found_evtgen)} / {len(COINC_EVENTS)}")
for run, event in COINC_EVENTS:
    present = ((df_evtgen["Run"] == run) & (df_evtgen["Event"] == event)).any()
    print(f"  Run={run}, Event={event}: {'FOUND' if present else 'NOT FOUND'}")

Coincident events found in EvtGen_merged.h5: 2 / 2
  Run=131680, Event=66412090: FOUND
  Run=132143, Event=36142391: FOUND


## 3. Summary table

In [11]:
rows = []
for run, event in COINC_EVENTS:
    for source, df in [("HESE12.hdf5", found_hese12), ("EvtGen_merged.h5", found_evtgen)]:
        match = df[(df["Run"] == run) & (df["Event"] == event)]
        if len(match) > 0:
            row = match.iloc[0].to_dict()
            row["Source"] = source
        else:
            row = {"Run": run, "Event": event, "RecoETot": np.nan, "Source": source + " (absent)"}
        rows.append(row)

df_summary = pd.DataFrame(rows)[["Source", "Run", "Event", "RecoETot", "QTot", "HESE_CausalQTot"]]
df_summary["RecoETot"]        = df_summary["RecoETot"].map(lambda x: f"{x:.1f}" if pd.notna(x) else "—")
df_summary["QTot"]            = df_summary["QTot"].map(lambda x: f"{x:.1f}" if pd.notna(x) else "—")
df_summary["HESE_CausalQTot"] = df_summary["HESE_CausalQTot"].map(lambda x: f"{x:.1f}" if pd.notna(x) else "—")

display(df_summary.set_index(["Source", "Run", "Event"]))

,,,RecoETot,QTot,HESE_CausalQTot
Source,Run,Event,,,
HESE12.hdf5,131680.0,66412090.0,201392.1,17244.4,—
EvtGen_merged.h5,131680.0,66412090.0,225256.3,—,15959.4
HESE12.hdf5,132143.0,36142391.0,190087.0,9295.7,—
EvtGen_merged.h5,132143.0,36142391.0,199360.8,—,6855.2
